# Kiểm thử Plant Disease AI trên Google Colab

Notebook này clone repository, cài dependencies, kiểm tra model, chạy giao diện Streamlit và tạo Cloudflare Quick Tunnel để mở ứng dụng từ trình duyệt. Có thêm bước kiểm thử dự đoán trực tiếp bằng ảnh tải lên Colab.

**Lưu ý:** URL Quick Tunnel là công khai và tạm thời. Không tải lên ảnh riêng tư hoặc dữ liệu nhạy cảm. Notebook này dùng để kiểm thử, không phải triển khai production.

In [ ]:
# 1. Clone/cập nhật repository và cài dependencies
import os, sys, subprocess, time, re, json
from pathlib import Path

REPO_DIR = Path("/content/Plant-Disease-Detection")
REPO_URL = "https://github.com/ShimeKano/Plant-Disease-Detection.git"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
print("Repository:", REPO_DIR)
print("Dependencies installed.")

In [ ]:
# 2. Kiểm tra model và cấu trúc ứng dụng
import tensorflow as tf

MODEL_DIR = REPO_DIR / "model"
MODEL_PATH = MODEL_DIR / ("best_model.keras" if (MODEL_DIR / "best_model.keras").exists() else "best_model.h5")
assert (REPO_DIR / "app.py").exists(), "Không tìm thấy app.py"
assert MODEL_PATH.exists(), f"Không tìm thấy model: {MODEL_PATH}"

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "Không có GPU; inference vẫn chạy trên CPU")
print("Model:", MODEL_PATH)
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print("Input:", model.input_shape)
print("Output:", model.output_shape)
print("Parameters:", f"{model.count_params():,}")
del model

In [ ]:
# 3. Khởi chạy Streamlit trong nền
import requests

APP_LOG = Path("/content/plant_disease_streamlit.log")
app_log_file = open(APP_LOG, "w")
streamlit_process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", "8501", "--server.address", "0.0.0.0", "--server.headless", "true"],
    cwd=str(REPO_DIR), stdout=app_log_file, stderr=subprocess.STDOUT
)

ready = False
for attempt in range(60):
    if streamlit_process.poll() is not None:
        raise RuntimeError("Streamlit đã thoát. Xem log: " + str(APP_LOG))
    try:
        response = requests.get("http://127.0.0.1:8501/_stcore/health", timeout=2)
        if response.ok:
            ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

if not ready:
    print(APP_LOG.read_text(errors="ignore")[-5000:])
    raise TimeoutError("Streamlit không khởi động trong thời gian chờ.")
print("Streamlit is healthy at http://127.0.0.1:8501")

In [ ]:
# 4. Tạo Cloudflare Quick Tunnel
import platform

cloudflared_path = "/usr/local/bin/cloudflared"
if platform.machine().lower() not in ("x86_64", "amd64"):
    raise RuntimeError("Notebook hiện tải cloudflared Linux x86_64; kiến trúc runtime: " + platform.machine())

subprocess.run([
    "bash", "-lc",
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
    "-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared"
], check=True)

TUNNEL_LOG = Path("/content/cloudflared.log")
tunnel_log_file = open(TUNNEL_LOG, "w")
tunnel_process = subprocess.Popen(
    [cloudflared_path, "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
    stdout=tunnel_log_file, stderr=subprocess.STDOUT
)

public_url = None
for attempt in range(60):
    time.sleep(2)
    log_text = TUNNEL_LOG.read_text(errors="ignore") if TUNNEL_LOG.exists() else ""
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    print(TUNNEL_LOG.read_text(errors="ignore")[-6000:])
    raise TimeoutError("Không lấy được URL Cloudflare. Kiểm tra log hoặc chạy lại cell.")

print("ỨNG DỤNG ĐÃ SẴN SÀNG:", public_url)
print("Mở URL trên để upload ảnh và kiểm thử giao diện.")

In [ ]:
# 5. Kiểm thử dự đoán trực tiếp bằng ảnh tải lên (tùy chọn)
from google.colab import files
from PIL import Image
import numpy as np
from io import BytesIO

uploaded = files.upload()
if uploaded:
    model = tf.keras.models.load_model(MODEL_PATH, compile=False)
    class_file = MODEL_DIR / "class_names.json"
    if class_file.exists():
        class_names = json.loads(class_file.read_text(encoding="utf-8"))
    else:
        class_names = [
            "Pepper__bell___Bacterial_spot", "Pepper__bell___healthy",
            "Potato___Early_blight", "Potato___Late_blight", "Potato___healthy",
            "Tomato__Bacterial_spot", "Tomato__Early_blight", "Tomato__Late_blight",
            "Tomato__Leaf_Mold", "Tomato__Septoria_leaf_spot",
            "Tomato__Spider_mites_Two_spotted_spider_mite", "Tomato__Target_Spot",
            "Tomato__Tomato_YellowLeaf__Curl_Virus", "Tomato__Tomato_mosaic_virus", "Tomato__healthy"
        ]

    for filename, content in uploaded.items():
        image = Image.open(BytesIO(content)).convert("RGB").resize((128, 128))
        batch = np.expand_dims(np.asarray(image, dtype=np.float32) / 255.0, axis=0)
        scores = model.predict(batch, verbose=0)[0]
        top = np.argsort(scores)[-3:][::-1]
        print("\n" + filename)
        for idx in top:
            label = class_names[int(idx)] if int(idx) < len(class_names) else f"class_{idx}"
            print(f"{label}: {scores[idx] * 100:.2f}%")
    del model
else:
    print("Không có ảnh được tải lên; bỏ qua kiểm thử dự đoán trực tiếp.")

In [ ]:
# 6. Dừng ứng dụng và tunnel khi kiểm thử xong
for process_name in ("tunnel_process", "streamlit_process"):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
        print(process_name, "stopped")
print("Done.")